In [1]:
# manual cleaning that was previously missed

import pandas as pd
import re
import numpy as np

df = pd.read_csv('new_master_dataset.csv')

In [2]:
# Impute state median income for states with no income tax

income_df = pd.read_csv('median-household-income-by-state-2026.csv')

income_cols = [
    'MedianHouseholdIncomeInOnePersonHousehold_2023',
    'MedianHouseholdIncomeInTwoPersonHousehold_2023',
    'MedianHouseholdIncomeInThreePersonHousehold_2023',
    'MedianHouseholdIncomeInFourPersonHousehold_2023'
]

no_tax_states = (
    income_df[income_df['state'].isin(
        ['Texas', 'Tennessee', 'Florida', 'Nevada']
        )
    ].copy()
)

no_tax_states['state_avg_median_income'] = no_tax_states[income_cols].mean(axis=1)

for _, row in no_tax_states.iterrows():
    mask = df['state_name'] == row['state']
    df.loc[mask , 'state_avg_median_income'] = row['state_avg_median_income']
# Verify
print(
    df[df['state_name'].isin(['Texas', 'Tennessee', 'Florida', 'Nevada'])]
    [['state_name', 'state_avg_median_income']]
    .drop_duplicates()
    .sort_values('state_name')
)

print(f"\nQA: Remaining NA in state_avg_median_income: {df['state_avg_median_income'].isnull().sum()}")

    state_name  state_avg_median_income
609    Florida                 79891.25
615     Nevada                 83007.25
617  Tennessee                 77656.75
122      Texas                 81742.50

QA: Remaining NA in state_avg_median_income: 0


In [3]:
# rows to remove
# 920	459883086	1861 Glasgo Road, Griswold, CT 06351 | Building is an old church, no bedrooms or bathrooms listed 
# 1297	235257090	2053 Pikes Falls Road, Jamaica, VT 05343 | Tiny home, SQFT (265) could skew results
# 1755	40981856	228 Tiger Creek Rd, Roan Mountain, TN 37687 | Auction starting price is listed ($10,000)
# 1781	41873062	3650 River Rd, Decatur, TN 37322 | Auction starting price listed ($1)

# 2097	22609053	1113 Woodland Dr, Charleston, WV 25302 | Gutted home, not sold for actual value

# 144	29054064	2717 Laurel Valley Ln, Arlington, TX 76006 | home is an auction, falsely labeled as 'House for sale'
# 2092	22388069	1598 Campbell Dr, Huntington, WV 25705 | Gutted home, not sold for actual value


In [4]:
print(f'Number of rows {df.shape[0]}')

Number of rows 2159


In [5]:
zpids_to_remove = [459883086, 235257090, 40981856, 41873062, 22609053, 29054064, 22388069]

df = df[~df['zpid'].isin(zpids_to_remove)]

print(f'Number of rows {df.shape[0]}')

Number of rows 2152


In [6]:
# 2031	1923497	330 11th Ave N, Hopkins, MN 55343 | price $1 --> $1,065,000
# 497	21912588	2916 Fairfield Dr, Edmond, OK 73012 | SQFT 0 --> SQFT 1451
# 498	21743321	3305 Fireside Cir, Norman, OK 73072 | SQFT 0 --> SQFT 2100
# 2201	460336976	215 Buffalo Creek Rd, Williamson, WV 25661 | SQFT 0 --> SQFT 1631
# 2167	304616329	1202 3rd Run Rd, Glenville, WV 26351 | SQFT 0 --> SQFT 2420 (Zillow home description)
# 524	22096033	14806 E Latimer St, Tulsa, OK 74116 | bathrooms 0 --> bathrooms 2

# 145	52107549	10210 Stidham Rd, Conroe, TX 77302 | status 'Auction --> 'House for sale' & price $100k --> $2,597,000

# Source for 2201: https://www.mystatemls.com/property/215-buffalo-creek-rd-wiliamson-wv-25661/11642390/

In [7]:
update_values  = {
    1923497: {'price': 1065000},
    21912588: {'area_sqft': 1451},
    21743321: {'area_sqft': 2100},
    460336976: {'area_sqft': 1631},
    304616329: {'area_sqft': 2420},
    22096033: {'baths': 2},
    52107549: {'status': 'House for sale', 'price': 2597000}

}

for zpid, updates in update_values.items():
    for col, val in updates.items():
        df.loc[df['zpid'] == zpid, col] = val

# Ensures no rows were deleted by accident (number should be the same as previous block)
print(f'Number of rows {df.shape[0]}')


Number of rows 2152


In [8]:
round(df.describe(), 4)

,Unnamed: 0,zpid,price,beds,baths,area_sqft,latitude,longitude,days_on_zillow,zestimate,state_median_housing_value,state_median_prop_tax_rate,med_prop_tax_paid,single_filer_rates,single_filer_brackets,married_filing_jointly_rates,married_filing_jointly_brackets,state_avg_median_income
count,2152.0000,2.152000e+03,2.152000e+03,2152.0000,2152.0000,2102.0000,2151.0000,2151.0000,2152.0000,1.084000e+03,2152.0000,2152.0000,2152.0000,1774.0000,1774.0000,1774.0000,1774.0000,2152.0000
mean,1099.1887,1.589022e+08,1.356538e+06,3.6877,2.9296,2715.1694,39.3890,-92.9936,23.0386,7.111603e+05,262372.4210,0.8923,2563.3734,5.6842,39581.3997,5.6264,65418.2802,88311.0983
std,640.0858,3.639104e+08,8.656892e+06,1.2174,1.6750,2800.3739,4.5863,16.1206,60.8659,1.543171e+06,107501.4777,0.5090,2011.7535,1.5460,32763.2537,1.5349,62882.4149,13356.4069
min,0.0000,8.239400e+04,3.550000e+04,1.0000,1.0000,572.0000,26.3919,-124.0970,-1.0000,2.920000e+04,134650.0000,0.2800,481.0000,2.5000,0.0000,2.5000,0.0000,65850.7500
25%,542.7500,2.288197e+07,3.390000e+05,3.0000,2.0000,1654.5000,35.9882,-105.0863,2.0000,3.471500e+05,169250.0000,0.5200,1134.0000,4.7000,7200.0000,4.7000,9436.0000,78990.2500
50%,1094.0000,5.258102e+07,4.950000e+05,4.0000,3.0000,2200.0000,39.5878,-90.6222,4.0000,4.908500e+05,231900.0000,0.6900,1888.0000,5.6500,33310.0000,5.5300,48700.0000,83007.2500
75%,1653.2500,1.138432e+08,7.605000e+05,4.0000,3.0000,2946.0000,42.5215,-78.7806,9.0000,7.428750e+05,335600.0000,1.0700,3293.5000,6.6000,72724.0000,6.6000,95000.0000,99511.7500
max,2213.0000,2.134250e+09,3.000000e+08,18.0000,27.0000,50000.0000,48.7907,-69.4295,745.0000,4.079420e+07,587550.0000,2.0300,8055.0000,9.3000,100000.0000,9.3000,200000.0000,114341.7500


In [9]:
# Dups removal
print(f'Number of rows {df.shape[0]}')

# finds dups
dup_mask = df.duplicated(subset='zpid', keep=False)

# for audit if needed later, unsure if dups were accidently added in somehow previously
df_dups = df[dup_mask].sort_values('zpid').copy()
df_dups.to_csv('dups_zpid.csv', index=False)

# drops dups
df = df.drop_duplicates(subset='zpid', keep='first').copy()

print(f'Number of rows {df.shape[0]}')


Number of rows 2152
Number of rows 2101


In [10]:
tax_cols = ['single_filer_rates', 'single_filer_brackets', 'married_filing_jointly_rates',
            'married_filing_jointly_brackets']

# fill missing with 0
# these the states that had missing state tax info do not have a state income tax (Texas, Tennessee, Nevada, etc.)
df[tax_cols] = df[tax_cols].fillna(0)


In [11]:
df[tax_cols].isnull().sum()

single_filer_rates                 0
single_filer_brackets              0
married_filing_jointly_rates       0
married_filing_jointly_brackets    0
dtype: int64

In [12]:
# save new dataset
df.to_csv('dataset_final.csv')
